<a href="https://colab.research.google.com/github/kaushikbaz123/Portfolio/blob/main/Project_1(stocks).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stock Momentum + Forecast Strategy — Forecast prices, generate trade signals, backtest, compute risk metrics (Sharpe, max drawdown), and recommend position sizing.

**How to use:** Run cells **top → bottom** in Google Colab. If a cell errors, copy the traceback and paste it back to me and I'll tell you the one-line fix.

**Notes**
- Avoid lookahead: targets and signals are shifted appropriately.
- Prediction horizon: 5 trading days.
- Train split ends on `2023-12-31`; test starts `2024-01-01`.
- Position sizing default: 10% per ticker; transaction cost default: 0.05% (0.0005).


In [8]:
!pip install yfinance pandas numpy matplotlib plotly scikit-learn xgboost ta==0.11.0

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor
import yfinance as yf
import ta
import joblib
plt.rcParams['figure.figsize'] = (10,6)

# create folders for outputs
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)


In [10]:
# ===== Replace your old download + selection with this block =====
tickers = ["AAPL","MSFT","AMZN"]
start = "2019-01-01"
end = "2025-12-01"

# ask yfinance to return auto-adjusted prices so there's no missing 'Adj Close'
df = yf.download(" ".join(tickers), start=start, end=end, progress=False, auto_adjust=True)

# pick the adjusted Close column robustly for multi-ticker or single-ticker outputs
if isinstance(df.columns, pd.MultiIndex):
    # For yfinance with auto_adjust=True and multiple tickers, columns are (Metric, Ticker)
    # e.g., ('Close', 'AAPL'), ('High', 'AAPL').
    # We want to select the 'Close' prices for all tickers.
    if 'Close' in df.columns.get_level_values(0): # Check if 'Close' metric exists at level 0
        price = df['Close'].copy() # Select the 'Close' prices for all tickers
        # The resulting 'price' DataFrame will have tickers as columns
    else:
        raise ValueError(f"No 'Close' metric found in MultiIndex columns. Available metrics: {sorted(set(df.columns.get_level_values(0)))}")
else:
    # single-ticker download case: map 'Close' to the expected ticker column name
    if 'Close' in df.columns:
        price = df[['Close']].copy()
        price.columns = [tickers[0]]
    else:
        raise ValueError(f"No 'Close' column found. Got columns: {df.columns}")

print("Downloaded price shape:", price.shape)
price.to_csv('data/finance_raw_prices.csv')
price.head()
# ===== end replacement =====

Downloaded price shape: (1738, 3)


Ticker,AAPL,AMZN,MSFT
Date,,,
2019-01-02,37.538822,76.956497,94.612602
2019-01-03,33.799671,75.014000,91.131989
2019-01-04,35.242561,78.769501,95.370476
2019-01-07,35.164116,81.475502,95.492111
2019-01-08,35.834454,82.829002,96.184494


In [11]:
def make_features(price_ser, fut_horizon=5):
    # price_ser: pd.Series of adjusted close indexed by date
    df = pd.DataFrame(index=price_ser.index)
    df['close'] = price_ser
    df['logret'] = np.log(df['close'] / df['close'].shift(1))

    # Moving averages and diffs
    for w in [5,10,20,50]:
        df[f'ma{w}'] = df['close'].rolling(w).mean()
        df[f'ma{w}_diff'] = (df['close'] - df[f'ma{w}']) / df[f'ma{w}']
    # Volatility (annualized)
    df['vol21'] = df['logret'].rolling(21).std() * np.sqrt(252)
    # RSI (14)
    df['rsi14'] = ta.momentum.rsi(df['close'], window=14)
    # Range (proxy for ATR since we only have close prices)
    df['range21'] = df['close'].rolling(21).apply(lambda x: x.max() - x.min(), raw=True)
    # Momentum
    df['mom10'] = df['close'] / df['close'].shift(10) - 1
    # Future 5-day return
    df['fut5ret'] = df['close'].shift(-fut_horizon) / df['close'] - 1
    df['target_dir'] = (df['fut5ret'] > 0).astype(int)
    df = df.dropna()
    return df

# Build per-ticker feature DataFrames and save
all_features = {}
for t in tickers:
    ser = price[t].dropna()
    all_features[t] = make_features(ser, fut_horizon=5)
    all_features[t].to_csv(f"data/{t}_features.csv")

In [12]:
train_end = '2023-12-31'
test_start = '2024-01-01'

def split_df(df):
    train = df.loc[:train_end].copy()
    test = df.loc[test_start:].copy()
    return train, test

train_test = {t: split_df(all_features[t]) for t in tickers}


In [13]:
FEATURES = ['ma5_diff','ma10_diff','ma20_diff','ma50_diff','vol21','rsi14','mom10']

models = {}
for t in tickers:
    train, test = train_test[t]
    X_train = train[FEATURES]
    y_train = train['target_dir']
    X_test = test[FEATURES]
    y_test = test['target_dir']

    model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)
    models[t] = model
    joblib.dump(model, f"models/{t}_xgb_clf.joblib")
    print(f"Trained and saved model for {t}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [13:48:34] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [13:48:34] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Trained and saved model for AAPL
Trained and saved model for MSFT
Trained and saved model for AMZN


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [13:48:34] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [14]:
from sklearn.metrics import classification_report
evals = {}
for t in tickers:
    train, test = train_test[t]
    X_test = test[FEATURES]
    y_test = test['target_dir']
    pred = models[t].predict(X_test)
    prob = models[t].predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred)
    report = classification_report(y_test, pred, output_dict=True)
    evals[t] = {'accuracy': acc, 'confusion_matrix': cm, 'report': report, 'prob': prob, 'y_test': y_test, 'X_test_idx': X_test.index}
    print(f"{t} — Accuracy: {acc:.4f}")
    print("Confusion matrix:\n", cm)
    print(classification_report(y_test, pred))

AAPL — Accuracy: 0.5011
Confusion matrix:
 [[ 59 148]
 [ 89 179]]
              precision    recall  f1-score   support

           0       0.40      0.29      0.33       207
           1       0.55      0.67      0.60       268

    accuracy                           0.50       475
   macro avg       0.47      0.48      0.47       475
weighted avg       0.48      0.50      0.48       475

MSFT — Accuracy: 0.5432
Confusion matrix:
 [[ 31 165]
 [ 52 227]]
              precision    recall  f1-score   support

           0       0.37      0.16      0.22       196
           1       0.58      0.81      0.68       279

    accuracy                           0.54       475
   macro avg       0.48      0.49      0.45       475
weighted avg       0.49      0.54      0.49       475

AMZN — Accuracy: 0.5221
Confusion matrix:
 [[ 77 129]
 [ 98 171]]
              precision    recall  f1-score   support

           0       0.44      0.37      0.40       206
           1       0.57      0.64      

In [15]:
def backtest_signals(df, prob, dates, prob_long=0.6, prob_short=0.4, position_size=0.10, tcost=0.0005):
    test_df = df.loc[dates].copy()
    test_df['prob'] = prob
    test_df['signal'] = 0
    test_df.loc[test_df['prob'] > prob_long, 'signal'] = 1
    test_df.loc[test_df['prob'] < prob_short, 'signal'] = -1
    test_df['signal_exec'] = test_df['signal'].shift(1).fillna(0)
    test_df['strategy_ret'] = test_df['signal_exec'] * test_df['logret'] * position_size
    test_df['pos_change'] = test_df['signal_exec'].diff().abs().fillna(0)
    test_df['tcost'] = test_df['pos_change'] * tcost * position_size
    test_df['strategy_ret_after_tc'] = test_df['strategy_ret'] - test_df['tcost']
    test_df['cum_strategy'] = test_df['strategy_ret_after_tc'].cumsum().apply(np.exp)
    return test_df

bt_results = {}
for t in tickers:
    train, test = train_test[t]
    prob = evals[t]['prob']
    dates = evals[t]['X_test_idx']
    bt = backtest_signals(all_features[t], prob, dates, prob_long=0.6, prob_short=0.4, position_size=0.10, tcost=0.0005)
    bt_results[t] = bt
    bt.to_csv(f"data/{t}_backtest.csv")
    print(f"Backtest finished for {t}: rows={len(bt)}")

Backtest finished for AAPL: rows=475
Backtest finished for MSFT: rows=475
Backtest finished for AMZN: rows=475


In [16]:
# Combine per-ticker strategy returns into a simple equal-weight portfolio (average)
strat_rets = pd.DataFrame({t: bt_results[t]['strategy_ret_after_tc'] for t in tickers})
strat_rets = strat_rets.dropna(how='all')
portfolio_ret = strat_rets.mean(axis=1)
portfolio_cum = (portfolio_ret.cumsum()).apply(np.exp)

portfolio_df = pd.DataFrame({
    'portfolio_ret': portfolio_ret,
    'portfolio_cum': portfolio_cum
})
portfolio_df.to_csv('data/portfolio_backtest.csv')
portfolio_df.head()

,portfolio_ret,portfolio_cum
Date,,
2024-01-02,0.000000,1.000000
2024-01-03,-0.000384,0.999616
2024-01-04,-0.000257,0.999360
2024-01-05,-0.000188,0.999172
2024-01-08,0.001386,1.000558


In [17]:
def annualized_sharpe(returns, periods_per_year=252):
    mean = returns.mean() * periods_per_year
    vol = returns.std() * np.sqrt(periods_per_year)
    return mean / vol if vol != 0 else np.nan

def cagr_from_returns(returns):
    total_return = np.exp(returns.cumsum().iloc[-1]) - 1
    days = returns.shape[0]
    years = days / 252.0
    return (1 + total_return) ** (1/years) - 1 if years>0 else np.nan

def max_drawdown(cum_returns):
    high = cum_returns.cummax()
    drawdown = (cum_returns - high) / high
    return drawdown.min()

def trade_stats_from_signals(bt_df):
    s = bt_df.copy()
    s['position'] = s['signal_exec']
    s['change'] = s['position'].diff()
    trades = []
    pos = 0
    entry_idx = None
    for idx, row in s.iterrows():
        if row['change'] != 0:
            if pos != 0 and entry_idx is not None:
                trade_ret = s.loc[entry_idx:idx]['strategy_ret_after_tc'].sum()
                trades.append(trade_ret)
                entry_idx = None
            if row['position'] != 0:
                entry_idx = idx
            pos = row['position']
    if pos != 0 and entry_idx is not None:
        trade_ret = s.loc[entry_idx:]['strategy_ret_after_tc'].sum()
        trades.append(trade_ret)
    trades = np.array(trades)
    if trades.size == 0:
        return {'num_trades':0, 'win_rate':np.nan, 'avg_trade_ret':np.nan}
    return {'num_trades': len(trades), 'win_rate': (trades>0).mean(), 'avg_trade_ret': trades.mean()}

port_returns = portfolio_df['portfolio_ret'].dropna()
port_cum = portfolio_df['portfolio_cum'].dropna()
port_cagr = cagr_from_returns(port_returns)
port_sharpe = annualized_sharpe(port_returns)
port_ann_vol = port_returns.std() * np.sqrt(252)
port_mdd = max_drawdown(port_cum)

print(f"Portfolio CAGR: {port_cagr:.2%}")
print(f"Annualized Volatility: {port_ann_vol:.2%}")
print(f"Sharpe (ann): {port_sharpe:.2f}")
print(f"Max Drawdown: {port_mdd:.2%}")

for t in tickers:
    stats = trade_stats_from_signals(bt_results[t])
    print(f"{t}: Trades={stats['num_trades']}, Win rate={stats['win_rate']}, Avg trade ret={stats['avg_trade_ret']}")

Portfolio CAGR: 0.65%
Annualized Volatility: 1.35%
Sharpe (ann): 0.48
Max Drawdown: -1.09%
AAPL: Trades=91, Win rate=0.5054945054945055, Avg trade ret=8.033092899912598e-05
MSFT: Trades=69, Win rate=0.5797101449275363, Avg trade ret=0.0003658070581749547
AMZN: Trades=110, Win rate=0.45454545454545453, Avg trade ret=1.2416682785192974e-05


In [18]:
# 1) price_vs_pred.png for AAPL
t = 'AAPL'
bt = bt_results[t]
fig, ax = plt.subplots()
ax.plot(all_features[t].loc[bt.index]['close'], label=f'{t} Close')
ax2 = ax.twinx()
ax2.plot(bt['prob'], alpha=0.9, label='pred_prob(5d)')
ax.set_title(f"{t} price and predicted prob (5-day)")
ax.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.savefig('price_vs_pred.png', bbox_inches='tight', dpi=150)
plt.close()

# 2) cum_returns.png (portfolio)
plt.figure()
plt.plot(portfolio_df['portfolio_cum'].fillna(method='ffill'))
plt.title('Portfolio cumulative returns')
plt.savefig('cum_returns.png', bbox_inches='tight', dpi=150)
plt.close()

# 3) drawdown.png
cum = portfolio_df['portfolio_cum'].fillna(method='ffill')
high = cum.cummax()
drawdown = (cum - high)/high
plt.figure()
plt.plot(drawdown)
plt.title('Portfolio drawdown')
plt.savefig('drawdown.png', bbox_inches='tight', dpi=150)
plt.close()

# 4) feature importance for AAPL
importances = models[t].feature_importances_
plt.figure()
plt.bar(FEATURES, importances)
plt.title(f'{t} Feature importance (XGB)')
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=150)
plt.close()

print('Saved price_vs_pred.png, cum_returns.png, drawdown.png, feature_importance.png')

/tmp/ipython-input-407570681.py:15: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  plt.plot(portfolio_df['portfolio_cum'].fillna(method='ffill'))
/tmp/ipython-input-407570681.py:21: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  cum = portfolio_df['portfolio_cum'].fillna(method='ffill')


Saved price_vs_pred.png, cum_returns.png, drawdown.png, feature_importance.png


In [19]:
# Save summary metrics and a README skeleton to paste numbers into
summary = {
    'portfolio_cagr': port_cagr,
    'portfolio_sharpe': port_sharpe,
    'portfolio_ann_vol': port_ann_vol,
    'portfolio_max_drawdown': port_mdd
}
pd.Series(summary).to_csv('data/summary_metrics.csv')

readme = f"""
# Stock Momentum + Forecast Strategy

## What this does
Forecasts next-5-day returns on AAPL, MSFT, AMZN using technical features, generates trade signals from predicted probabilities, backtests a fixed-fraction strategy and reports risk metrics.

## Data
Source: yfinance (Adj Close), 2019-01-01 to 2025-12-01.

## Models
XGBoost classifier for direction (probability -> signal).
Features: {FEATURES}

## Backtest config
- Prediction horizon: 5 trading days
- Train: up to 2023-12-31
- Test: from 2024-01-01
- Long threshold: prob > 0.6, Short threshold: prob < 0.4
- Position sizing: 10% per ticker
- Transaction cost: 0.05% (0.0005) per trade

## Key metrics (paste exact numbers here after running)
- Directional accuracy (per ticker): ...
- CAGR (portfolio): ...
- Annualized Volatility (portfolio): ...
- Sharpe (annual): ...
- Max Drawdown: ...
- Win rate, avg trade return, number of trades (per ticker): ...

## Business recommendation (2 lines)
Model shows [paste: positive/negative] expectancy on test (CAGR = X%, Sharpe = Y).
Recommendation: allocate X% of portfolio (or 10% per ticker) with stop loss = Y% and review monthly; start paper trading for 3 months before real capital.
"""

with open('README_PROJECT.md','w') as f:
    f.write(readme)
print('README_PROJECT.md created in working directory. Metrics CSV saved to data/summary_metrics.csv')

README_PROJECT.md created in working directory. Metrics CSV saved to data/summary_metrics.csv


## Checklist (run cells top → bottom)
1. Install dependencies (first cell)  
2. Download data (3rd cell)  
3. Feature creation (4th cell)  
4. Train/test split (5th cell)  
5. Train models (6th cell)  
6. Evaluate (7th cell)  
7. Backtest (8th cell)  
8. Portfolio combine & metrics (9th + 10th cells)  
9. Plots saved (11th cell)  
10. README + summary saved (12th cell)

**Next steps (recommended):**
- Paper-trade for 3 months before committing real capital.
- Implement walk-forward retraining (expanding-window) — I can add that if you want.
- Add slippage modeling and volume-based filters for more realism.
